In [1]:
import sys, os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from tqdm.notebook import tqdm

tqdm.pandas()

### Loading package

In [3]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [4]:
from py.utils import verifyDir,verifyFile

In [5]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/media/felipe/DATA21/datasets/', '/media/felipe/DATA21/models/')

### Loading data

In [6]:
DATA_PATH=f"{cfg.DATA_PATH}crimebb/"
SQL_PATH = f"{DATA_PATH}/{cfg.YEAR}/sql/"
CSV_PATH = f"{DATA_PATH}/{cfg.YEAR}/csv/"
CSV_SUMMARY = f"{CSV_PATH}/summary_from_sql/"
CSV_PROCESSED = f"{CSV_PATH}/processed_from_csv/"
WEBSITES_PATH = f"{CSV_PATH}/websites/"

In [7]:
verifyDir(CSV_PROCESSED)
verifyDir(WEBSITES_PATH)

### Processing data

In [8]:
from py.datasets import CrimeBB

crimeBB_manager = CrimeBB(data_path=CSV_PATH, year=cfg.YEAR, log=True)

#### Members

In [10]:
%%time
crimeBB_manager.read_and_process_members()

Reading directly ...
CPU times: user 6.21 s, sys: 1.16 s, total: 7.36 s
Wall time: 7.36 s


#### Boards & Websites

In [11]:
%%time
if cfg.YEAR in [2021, 2023]:
    crimeBB_manager.read_and_process_boards()
    crimeBB_manager.read_and_process_sites()
else:
    crimeBB_manager.read_and_process_sites()
    crimeBB_manager.read_and_process_boards()

Reading directly ...
Reading directly ...


#### Threads

In [12]:
%%time
crimeBB_manager.read_and_process_threads()    

Reading directly ...
CPU times: user 28.2 s, sys: 6.31 s, total: 34.5 s
Wall time: 34.5 s


#### Posts

In [ ]:
# crimeBB_manager.posts_path

In [ ]:
# #current_reader = crimeBB_manager.read_posts(read_direct=False)
# current_reader = pd.read_csv(crimeBB_manager.posts_path, sep="\t", low_memory=False, iterator=True, encoding='utf-8', on_bad_lines='skip')

In [ ]:
# chunk_size=1000000
# len_readed=chunk_size
# count=0
# while len_readed>=chunk_size:
#     current_df = current_reader.get_chunk(chunk_size)#.copy()
#     current_df.drop_duplicates(inplace=True)

#     len_readed = current_df.shape[0]
#     print(len_readed, count, chunk_size)
#     count+=1

In [13]:
%%time
crimeBB_manager.read_and_process_posts()

Reading by iterator ...
count: 0 readed: 1000000 chunck size: 1000000
count: 1 readed: 1000000 chunck size: 1000000
count: 2 readed: 1000000 chunck size: 1000000
count: 3 readed: 1000000 chunck size: 1000000
count: 4 readed: 1000000 chunck size: 1000000
count: 5 readed: 1000000 chunck size: 1000000
count: 6 readed: 1000000 chunck size: 1000000
count: 7 readed: 1000000 chunck size: 1000000
count: 8 readed: 1000000 chunck size: 1000000
count: 9 readed: 1000000 chunck size: 1000000
count: 10 readed: 1000000 chunck size: 1000000
count: 11 readed: 1000000 chunck size: 1000000
count: 12 readed: 1000000 chunck size: 1000000
count: 13 readed: 1000000 chunck size: 1000000
count: 14 readed: 1000000 chunck size: 1000000
count: 15 readed: 1000000 chunck size: 1000000
count: 16 readed: 1000000 chunck size: 1000000
count: 17 readed: 1000000 chunck size: 1000000
count: 18 readed: 1000000 chunck size: 1000000
count: 19 readed: 1000000 chunck size: 1000000
count: 20 readed: 1000000 chunck size: 1000000

### Summarizing CrimeBB

In [14]:
crimeBB_manager.get_process_time()

630.5917992591858

In [16]:
%%time
crimebb_df = crimeBB_manager.summarize()
crimebb_df.shape

CPU times: user 6.91 s, sys: 6.65 s, total: 13.6 s
Wall time: 13.6 s


(54443387, 12)

In [ ]:
#crimebb_df = crimebb_df[~crimebb_df["content"].isna()].copy()

In [17]:
crimebb_df.drop_duplicates(inplace=True)
crimebb_df.shape

(54443387, 12)

In [18]:
crimebb_df.isnull().any()

post_id               False
site_id               False
board_id              False
thread_id             False
user_id               False
site_name             False
board_title           False
thread_title           True
username               True
content                True
user_reputation       False
post_data_creation    False
dtype: bool

### Saving Final File

In [19]:
crimebb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54443387 entries, 0 to 54443386
Data columns (total 12 columns):
 #   Column              Dtype
---  ------              -----
 0   post_id             int64
 1   site_id             int64
 2   board_id            int64
 3   thread_id           int64
 4   user_id             int64
 5   site_name           str  
 6   board_title         str  
 7   thread_title        str  
 8   username            str  
 9   content             str  
 10  user_reputation     int64
 11  post_data_creation  str  
dtypes: int64(6), str(6)
memory usage: 22.2 GB


In [21]:
crimebb_df.head()

,post_id,site_id,board_id,thread_id,user_id,site_name,board_title,thread_title,username,content,user_reputation,post_data_creation
0,24355,3,19,5,22290,offensivecommunity.net/index.php,Advanced Hacking Tutorials,Wireless Hacking {Cracking WEP key},NaN,"thanks, for this great tutorial!",0,2015-12-25 09:56:00-02
1,23917,3,19,5,22115,offensivecommunity.net/index.php,Advanced Hacking Tutorials,Wireless Hacking {Cracking WEP key},NaN,***CITING***[http://offensivecommunity.net/pos...,0,2015-12-13 08:14:00-02
2,12023,3,19,5,19519,offensivecommunity.net/index.php,Advanced Hacking Tutorials,Wireless Hacking {Cracking WEP key},NaN,***CITING***[http://offensivecommunity.net/pos...,0,2015-07-23 23:18:00-03
3,2,3,19,5,27,offensivecommunity.net/index.php,Advanced Hacking Tutorials,Wireless Hacking {Cracking WEP key},NaN,It's good to see such a valuable information o...,0,2012-06-30 08:49:00-03
4,8,3,19,5,26,offensivecommunity.net/index.php,Advanced Hacking Tutorials,Wireless Hacking {Cracking WEP key},NaN,Perfect one!!!\neasy to learn with such kind o...,0,2012-06-30 07:48:00-03


In [22]:
crimebb_df.to_csv(f"{CSV_PATH}crimeBB.csv", sep='\t', index=False)

### Saving websites

In [23]:
websites = crimebb_df["site_name"].unique()
len(websites), np.sort(websites)

(16,
 array(['dreadditevelidot.onion/discover',
        'envoys5appps3bin.onion/index.php', 'forum.antichat.ru',
        'garage4hackers.com/forum.php', 'germanyruvvy2tcw.onion',
        'greysec.net/index.php', 'hackforums.net',
        'lwplxqzvmgu43uff.onion/index.php',
        'offensivecommunity.net/index.php', 'stresserforums.net/index.php',
        'thehub7xbw4dc5r2.onion/', 'torum6uvof666pzw.onion/index.php',
        'www.kernelmode.info/forum', 'www.mpgh.net', 'www.raidforums.com',
        'www.safeskyhacks.com/Forums/forum.php'], dtype=object))

In [24]:
for web_name_ in websites:
    web_name = web_name_.replace("www.", "")
    web_name = web_name.replace("forum.", "")
    web_name = web_name.replace("foro.", "")
    web_name = web_name.replace(".ru", "")
    web_name = web_name.replace(".io", "")
    web_name = web_name.replace(".info", "")
    web_name = web_name.replace(".com", "")
    web_name = web_name.replace(".onion", "")
    web_name = web_name.replace(".biz", "")
    web_name = web_name.replace(".world", "")
    web_name = web_name.replace(".online", "")
    web_name = web_name.replace(".fun", "")
    web_name = web_name.replace(".cf", "")
    web_name = web_name.replace(".net", "")
    web_name = web_name.replace(".org", "")
    web_name = web_name.replace(".vc", "")
    web_name = web_name.replace(".to", "")
    web_name = web_name.replace(".info", "")
    web_name = web_name.replace(".one", "")
    web_name = web_name.replace(".is", "")
    web_name = web_name.replace(".me", "")
    web_name = web_name.replace("/index.php", "")
    web_name = web_name.replace("/forum", "")
    web_name = web_name.replace("/discover", "")
    web_name = web_name.replace("/php", "")
    web_name = web_name.replace("/Forums", "")
    web_name = web_name.replace("/", "")
    if web_name in ["lolzteam", "forumteam", "cracked", "kernelmode"]:
        continue
    print(web_name_, web_name)

    df_ = crimebb_df[crimebb_df["site_name"]==web_name_].copy()
    df_.to_csv(f"{WEBSITES_PATH}{web_name}.csv", sep="\\", index=False)

offensivecommunity.net/index.php offensivecommunity
torum6uvof666pzw.onion/index.php torum6uvof666pzw
www.raidforums.com raidforums
stresserforums.net/index.php stresserforums
www.mpgh.net mpgh
greysec.net/index.php greysec
thehub7xbw4dc5r2.onion/ thehub7xbw4dc5r2
forum.antichat.ru antichat
envoys5appps3bin.onion/index.php envoys5appps3bin
garage4hackers.com/forum.php garage4hackers
dreadditevelidot.onion/discover dreadditevelidot
www.safeskyhacks.com/Forums/forum.php safeskyhacks
lwplxqzvmgu43uff.onion/index.php lwplxqzvmgu43uff
germanyruvvy2tcw.onion germanyruvvy2tcw
hackforums.net hackforums


In [25]:
for web_name_ in [['lolzteam.online', 'lolzteam.net'], ['forumteam.world', 'forumteam.fun'], ['cracked.io', 'cracked.to'], ['kernelmode.info', 'www.kernelmode.info']]:
    df_new = pd.DataFrame()
    for wb_ in web_name_:
        web_name = wb_.replace("www.", "")
        web_name = web_name.replace("forum.", "")
        web_name = web_name.replace("foro.", "")
        web_name = web_name.replace(".ru", "")
        web_name = web_name.replace(".io", "")
        web_name = web_name.replace(".info", "")
        web_name = web_name.replace(".com", "")
        web_name = web_name.replace(".onion", "")
        web_name = web_name.replace(".biz", "")
        web_name = web_name.replace(".world", "")
        web_name = web_name.replace(".online", "")
        web_name = web_name.replace(".fun", "")
        web_name = web_name.replace(".cf", "")
        web_name = web_name.replace(".net", "")
        web_name = web_name.replace(".org", "")
        web_name = web_name.replace(".vc", "")
        web_name = web_name.replace(".to", "")
        web_name = web_name.replace(".info", "")
        web_name = web_name.replace(".one", "")
        web_name = web_name.replace(".is", "")
        web_name = web_name.replace(".me", "")
        web_name = web_name.replace("/index.php", "")
        web_name = web_name.replace("/forum", "")
        web_name = web_name.replace("/discover", "")
        web_name = web_name.replace("/php", "")
        web_name = web_name.replace("/Forums", "")
        web_name = web_name.replace("/", "")
        
        print(wb_, web_name)

        df_ = crimebb_df[crimebb_df["site_name"]==wb_].copy()
        df_new = pd.concat([df_new, df_], ignore_index=True)
        
    df_new.to_csv(f"{WEBSITES_PATH}{web_name}.csv", sep="\\", index=False)

lolzteam.online lolzteam
lolzteam.net lolzteam
forumteam.world forumteam
forumteam.fun forumteam
cracked.io cracked
cracked.to cracked
kernelmode.info kernelmode
www.kernelmode.info kernelmode
